In [1]:
import cv2
from pathlib import Path
from PIL import Image
import pandas as pd
import sys

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import albumentations as A
import os
from glob import glob

In [2]:
sys.path.append('../..')

%load_ext autoreload
%autoreload 2

In [3]:
def filter_dataset(
        df,
        path_to_dataset,
        class_names_to_remove=["yeezy_slide"],
        bad_images_md_path="../2-exploration/bad_images.md"
):
    """

    """
    # Вычищаем Yeezy Slide, т.к. они не укладываются в наши представления о кроссовках
    df = df.query('sneaker_class != "yeezy_slide"')
    # Фильтруем битые картинки, рисунки и т.п.
    # Плохие картинки мы отсмотрели вручную. Список можно увидеть в файле bad_images
    with open(bad_images_md_path) as f:
        bad_images = f.read().strip().split("\n")

    bad_image_paths = [
        image[image.find("(") + 1 : -1] for image in bad_images if len(image) > 0
    ]  # В md указаны пути в формате ![name](path). Поэтому путь это все, что находится в круглых скобках ()

    bad_image_paths = [
        os.path.relpath(
            image, start=path_to_dataset
        )  # Тут заменил, надо заменить пути в bad_images.md
        for image in bad_image_paths
    ]

    print(f"Bad images len: {len(bad_image_paths)}")

    df = df[df["path"].apply(lambda x: x not in bad_image_paths)]
    
    print(f"Dataframe size: {df.shape[0]}")

    return df

In [4]:
# Изображения хранятся в директории так, что каждой модели кроссовок соответствует
# своя папка. Чтобы можно было рассчитывать статистики, мы собираем в датафрейм относительный
# путь до каждого из изображений, и рассчитываем агрегаты на основе этого датафрейма
from src.data.utils.eda_utils import directory_to_dataframe

path_to_dataset = Path("../../data/01_raw/sneakers-dataset")
df = directory_to_dataframe(path_to_dataset)
# Вычищаем Yeezy Slide, т.к. они не укладываются в наши представления о кроссовках
df = filter_dataset(
    df,
    path_to_dataset
)

df.head()

Bad images len: 12
Dataframe size: 5796


,path,sneaker_class
0,reebok_classic_leather/0071.jpg,reebok_classic_leather
1,reebok_classic_leather/0065.jpg,reebok_classic_leather
2,reebok_classic_leather/0059.jpg,reebok_classic_leather
3,reebok_classic_leather/0058.jpg,reebok_classic_leather
4,reebok_classic_leather/0064.jpg,reebok_classic_leather


In [5]:
# Пайплайн аугментаций Albumentations
augmenter = A.Compose([
    A.RandomBrightnessContrast(p=0.5),
    A.GaussianBlur(p=0.2),
    A.MotionBlur(p=0.2),
    A.OneOf([
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
        A.OpticalDistortion(p=0.3),
        A.GridDistortion(p=0.3),
    ], p=0.7),
    A.CoarseDropout(max_holes=1, max_height=16, max_width=16, p=0.3)
])

/Users/azat/vscode_projects/sneakers-hse/.env/lib/python3.12/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/var/folders/sj/99tdnx5d73l54szyyxy3s3tc0000gn/T/ipykernel_27635/2785960658.py:11: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=1, max_height=16, max_width=16, p=0.3)


In [6]:
# Функция чтения и аугментации изображений
def load_image(img_path): #, augmenter):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)  # градации серого для классических признаков
    if img is None:
        raise FileNotFoundError(f"Image not found: {img_path}")
    # augmented = augmenter(image=img)
    # return augmented['image']
    return img


# Функция для вычисления признаков HOG
def extract_hog_features(image):
    winSize = (64, 64)
    blockSize = (16, 16)
    blockStride = (8, 8)
    cellSize = (8, 8)
    nbins = 9
    hog = cv2.HOGDescriptor(winSize, blockSize, blockStride, cellSize, nbins)
    img_resized = cv2.resize(image, winSize)
    features = hog.compute(img_resized).flatten()
    return features


In [19]:
from sklearn.base import TransformerMixin, BaseEstimator

class HogTransformer(TransformerMixin, BaseEstimator):

    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        return extract_hog_features(X)
    
    def fit_transform(self, X):
        return self.transform(X)


In [7]:
# Загрузка данных из папок с классами и извлечение признаков
def load_dataset(
    base_path,
    images_path,
    class_names,
    augmenter
):
    X, y = [], []
    for img_file, label_img in zip(images_path, class_names):
        try:
            img = load_image(base_path / img_file)

            features = extract_hog_features(img)
            X.append(features)
            y.append(label_img)
        except Exception as e:
            print(f"Skipping image {base_path / img_file} due to error: {e}")
    return np.array(X), np.array(y)

In [8]:
# Путь к вашей датасете с папками по классам с jpg
images_path = df['path'].to_list()
class_names = df['sneaker_class'].to_list()

# Загрузка и аугментация
X, y = load_dataset(
    path_to_dataset,
    images_path,
    class_names,
    augmenter
)


In [9]:
X.shape

(5796, 1764)

In [10]:
# Делим на train+val и test, а потом train и val
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, stratify=y_train_val, random_state=42
)  # 0.25*0.8=0.2

In [12]:
# Считаем веса классов для балансировки
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_number_to_class_name = dict(enumerate(np.unique(y_train)))
class_weights_dict = {class_number_to_class_name[i]: w for i, w in enumerate(class_weights)}

## SVC

In [13]:
# Масштабирование признаков
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [14]:
X_train.shape

(3477, 1764)

In [15]:
X_train

array([[ 0.9151478 ,  1.0870912 ,  0.8627137 , ...,  2.313168  ,
         2.6005056 ,  1.1464826 ],
       [-0.6540305 , -0.564558  , -0.58091646, ..., -0.6102329 ,
        -0.5773474 , -0.6817775 ],
       [ 0.46978393,  0.8514592 ,  0.9140655 , ..., -0.55289024,
        -0.5566759 , -0.66999984],
       ...,
       [ 2.2087843 ,  0.13007638,  1.3553501 , ..., -0.6053247 ,
        -0.57556444, -0.63243103],
       [-0.6540305 , -0.564558  , -0.58091646, ..., -0.5825346 ,
        -0.54360217, -0.6066792 ],
       [-0.6540305 , -0.564558  , -0.58091646, ..., -0.6102329 ,
        -0.5773474 , -0.6817775 ]], shape=(3477, 1764), dtype=float32)

In [16]:
y_train

array(['converse_chuck_70_low', 'nike_dunk_low', 'vans_old_skool', ...,
       'nike_air_jordan_1_low', 'converse_chuck_70_high', 'salomon_xt-6'],
      shape=(3477,), dtype='<U35')

In [17]:
# Обучение SVM с учетом весов классов
svm_clf = SVC(class_weight=class_weights_dict)
svm_clf.fit(X_train, y_train)
y_val_pred = svm_clf.predict(X_val)
print("SVM Validation Classification Report:")
print(classification_report(y_val, y_val_pred))

SVM Validation Classification Report:
                                     precision    recall  f1-score   support

                  adidas_forum_high       0.19      0.34      0.24        29
                   adidas_forum_low       0.29      0.22      0.25        18
                     adidas_gazelle       0.50      0.33      0.40        30
                      adidas_nmd_r1       0.67      0.32      0.43        19
                       adidas_samba       0.17      0.07      0.10        15
                  adidas_stan_smith       0.34      0.34      0.34        29
                   adidas_superstar       0.33      0.17      0.22        18
                  adidas_ultraboost       0.43      0.43      0.43        30
                 asics_gel-lyte_iii       0.69      0.50      0.58        18
             converse_chuck_70_high       0.31      0.33      0.32        15
              converse_chuck_70_low       0.18      0.27      0.21        30
converse_chuck_taylor_all-star_high  

In [20]:
from sklearn.pipeline import Pipeline

In [27]:
pipe = Pipeline([
    ('scaler', StandardScaler),
    ('svc', svm_clf)
])

In [35]:
extract_hog_features(img).reshape(1, -1)

array([[0.10851903, 0.12449397, 0.07661647, ..., 0.1512266 , 0.20538795,
        0.22529103]], shape=(1, 1764), dtype=float32)

In [46]:
svm_clf.predict(scaler.transform(
    X=extract_hog_features(img).reshape(1, -1)
))[0]

np.str_('nike_air_jordan_4')

In [24]:

path_to_dataset / df.iloc[0, 0]

PosixPath('../../data/01_raw/sneakers-dataset/reebok_classic_leather/0071.jpg')

In [42]:
img = load_image(path_to_dataset / df.iloc[0, 0])


In [43]:
type(img)

numpy.ndarray

In [18]:
import pickle

with open("../../models/baseline_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

with open("../../models/svc.pkl", "wb") as f:
    pickle.dump(svm_clf, f)




## catboost

In [75]:
# Обучение CatBoost с весами классов
cat_clf = CatBoostClassifier(iterations=100, eval_metric='MultiClass', class_weights=class_weights.tolist(), random_seed=42)
cat_clf.fit(X_train, y_train, eval_set=(X_val, y_val))
y_val_pred_cat = cat_clf.predict(X_val)
print("CatBoost Validation Classification Report:")
print(classification_report(y_val, y_val_pred_cat))

Learning rate set to 0.266779
0:	learn: 3.8111355	test: 3.8540872	best: 3.8540872 (0)	total: 3.41s	remaining: 5m 37s
1:	learn: 3.7437267	test: 3.8266003	best: 3.8266003 (1)	total: 6.74s	remaining: 5m 30s
2:	learn: 3.6574813	test: 3.7957120	best: 3.7957120 (2)	total: 10.3s	remaining: 5m 33s
3:	learn: 3.5893853	test: 3.7744791	best: 3.7744791 (3)	total: 13.9s	remaining: 5m 33s
4:	learn: 3.5416993	test: 3.7657386	best: 3.7657386 (4)	total: 17.4s	remaining: 5m 30s
5:	learn: 3.4798389	test: 3.7420774	best: 3.7420774 (5)	total: 20.8s	remaining: 5m 26s
6:	learn: 3.4255842	test: 3.7313740	best: 3.7313740 (6)	total: 24.3s	remaining: 5m 23s
7:	learn: 3.3674921	test: 3.7215900	best: 3.7215900 (7)	total: 27.9s	remaining: 5m 21s
8:	learn: 3.3223798	test: 3.7026243	best: 3.7026243 (8)	total: 31.4s	remaining: 5m 17s
9:	learn: 3.2746510	test: 3.6874231	best: 3.6874231 (9)	total: 34.8s	remaining: 5m 13s
10:	learn: 3.2120172	test: 3.6736127	best: 3.6736127 (10)	total: 38.3s	remaining: 5m 10s
11:	learn: 